Notebook examines parameter recapitulation using synthetic data.

In [1]:
import sys
import os
import scMPRAforge as scm
import pandas as pd
import numpy as np

2025-07-26 16:43:00.766982: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-26 16:43:00.771056: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2025-07-26 16:43:00.771069: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.


In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
from dask.distributed import Client, LocalCluster
cluster=LocalCluster()
client = Client(cluster)

In [4]:
dat=scm.scMPRA_data.from_tsv("/gpfs/gibbs/pi/reilly/tabula_data/simulated/fake_cres_v3.tsv")
dat.ortho_filter()
dat.set_negative_controls(["nobody","weak"])
dat.set_reference_cell("liver")

primordial=scm.ortho()
primordial.criss_cross(client=client,dat=dat)
primordial.extract_params(client)

batch=scm.simulation_batch(primordial)
batch.describe_primordial()

scMPRAforge: INFO: Dropped 0 of 27 (cell_type, cre_id) combos with fewer than 3 nonzero entries.


In [22]:
by_cell_type=batch.description_primordial_by_cell_type.compute()
by_cell_type=by_cell_type.set_index(['cell_type','cre_id'])
by_cell_type.head()

index rep_id          mu        zi     theta  cells  \
cell_type cre_id                                                            
blood     everybody       0      1  106.800573  0.100628  3.323221    489   
          everybody       1      2  106.800573  0.202129  3.323221    514   
          everybody       2      3  106.800573  0.305679  3.323221    496   
          hepatogene      3      1   14.927022  0.100628  3.323221    516   
          hepatogene      4      2   14.927022  0.202129  3.323221    523   

                             r  sigmasquare         p  
cell_type cre_id                                       
blood     everybody   3.323221  3539.121681  0.030177  
          everybody   3.323221  3539.121681  0.030177  
          everybody   3.323221  3539.121681  0.030177  
          hepatogene  3.323221    81.975223  0.182092  
          hepatogene  3.323221    81.975223  0.182092

In [25]:
ground_truth_mu=pd.read_csv("../../notebooks/demos/parameter_extraction_demo/synthetic_ground_truth.tsv",sep="\t")
ground_truth_mu=ground_truth_mu.rename({'CRE':'cre_id','mean':'true_mu','Cell-type':'cell_type'},axis=1)
ground_truth_mu=ground_truth_mu.set_index(['cell_type','cre_id'])
ground_truth_mu.head()

true_mu
cell_type cre_id             
brain     nobody            1
          mediumbody       10
          everybody       114
          redgene          30
          neurogene        99

In [ ]:
comp_df=by_cell_type.

,cre_id,Cell-type,true_mu
0,nobody,brain,1
1,mediumbody,brain,10
2,everybody,brain,114
3,redgene,brain,30
4,neurogene,brain,99
